# Stratum — a guided tour

Stratum is a playground for the **reflective ρ-calculus** as a protocol design & analysis language:
write a protocol in a small surface syntax, reduce it to a *trace* (a labelled transition system),
and check temporal / equivalence properties over that trace.

**To run this notebook:** pick the **Stratum** kernel (top-right → *Select Kernel* → *Jupyter Kernel…* → *Stratum*).

A cell is either a `.strat` **DSL** cell (a process, optionally bound with `name = …`),
a `%`-**directive**, or a `%%rune` **script**. `%help` lists every directive.

## 1. Define a protocol

A one-shot **request / acknowledge** handshake. `new req, ack` mints two *distinct* fresh channel
names; the client sends on `req`, and the server receives it (binding the reply value `x`, unused
here) and answers on `ack`. We bind the whole process to `hs`.

Running the cell echoes the process in surface form (with the received binder shown as `v0`).

In [ ]:
hs = new req, ack

req!(0) | req(x).ack!(0)

## 2. Explore the trace

`%explore` builds the bounded trace LTS and renders it inline as an SVG (states are canonical
processes; edges are tagged with the channel the `Comm` fired on). `-> g` binds the LTS to `g`
so later cells can refer to it.

In [ ]:
%explore hs -> g

## 3. Check temporal properties

Formulas are modal μ-calculus (subsuming LTL/CTL): `EF/AG/AF/EG/EX`, `&` `|` `!`, and the atomic
proposition `emits(<channel>)` — a top-level output *barb* on that channel.

- `EF emits(ack)` — *the acknowledgement is reachable*.
- `AG emits(ack)` — *ack is emitted in every state* (it is not: the initial state hasn't acked yet).

In [ ]:
%check EF emits(ack) on g

In [ ]:
%check AG emits(ack) on g

## 4. Extract a run

`%witness <goal>` returns the shortest run reaching a state satisfying the goal — here, the step
that produces the `ack` barb. (`%counterexample <invariant>` is the dual, for safety invariants.)

In [ ]:
%witness emits(ack) on g

## 5. Transparency — desugar to the core

The surface syntax is thin sugar over the pure ρ-calculus. `%expand` shows the raw core the sugar
elaborates to: `new` names become concrete quoted-process names (`req = @0`, `ack = @(@0!(0))`),
and the received binder becomes a de Bruijn `v0`. Nothing is hidden.

In [ ]:
%expand hs

## 6. Behavioural equivalence

`%bisim` decides N-barbed bisimulation (parameterised by an observation set, defaulting to every
channel in either process). An *emitter* and a *silent input* on the same channel differ on their
observable barbs, so they are distinguished; a process is of course equivalent to itself.

In [ ]:
emitter = @0!(0)

In [ ]:
quiet = @0(x).0

In [ ]:
%bisim emitter quiet

## 7. Custom algorithms with `%%rune`

A `%%rune` cell drops into an embedded scripting VM with the toolkit bound in (`explore`, `check`,
`bisim`, `step`, …) and the session namespace shared (`get`/`set`). Write your own analyses over the
*live* objects without leaving the notebook. The final value renders like any other result.

In [ ]:
%%rune
use stratum::*;

let lts = explore(get("hs"), 100);
println(`states: ${lts.num_states()}, transitions: ${lts.num_transitions()}, truncated: ${lts.is_truncated()}`);

let terminals = 0;
for i in 0..lts.num_states() {
    if lts.successors(i).is_empty() { terminals += 1; }
}
println(`terminal (deadlock) states: ${terminals}`);

lts

## Where next

`%help` lists every directive. Beyond the above, Stratum also has:

- `%explore … por` / `sym=a,b` — partial-order & symmetry-reduced exploration.
- `%check` with epistemic `K_A`/`P_A` and fairness; on-the-fly reachability with early exit.
- `%typecheck` — a channel-sort typing discipline.
- labelled bisimulation / barbed congruence in the toolkit.

The core reduction and structural congruence are machine-checked in Coq (`proofs/`).

In [ ]:
%help